In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

In [ ]:
#read Data
Data = pd.read_csv("/content/Q1_data.csv")
Data

In [ ]:

#Print head
Data = pd.read_csv("/content/Q1_data.csv")
Data.head()

In [ ]:
# Task 3: Write your code here:

# Check data types and structure
Data.info()

In [ ]:
# Task 4: Write your code here:

# Descriptive statistics for numerical columns
Data.describe()

In [ ]:
# Task 5: Write your code here:
#Plot Delivery Time
condition_counts = Data['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(condition_counts.index, condition_counts.values, color='coral')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop Order ID
Data = Data.drop("Order_ID",axis=1)

In [ ]:
# Task 2: Write your code here:

#check missing value
print(Data.isnull().sum())

In [ ]:
# Handle missing values
# we have Weather, Traffic_Level, Time_of_Day, Courier_Experience_yrs and Delivery_Time have null
Data = Data.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])

In [ ]:
# Task 3: Write your code here:
#check duplicated value
Data.duplicated().sum()

# drop duplicated
Data = Data.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
# Encode variables by OneHotEncoder
print('data before encoding:\n', Data) # before encoding

onehot_encoder = OneHotEncoder(sparse_output=False)
data_onehot_encoded = onehot_encoder.fit_transform(Data)

print('\nData after encoding:\n', data_onehot_encoded) # after encoding

In [ ]:
# Task 5: Write your code here:
# feature scaling by StandardScaler
numerical_cols = Data.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # else target

scaler = StandardScaler()
Data[numerical_cols] = scaler.fit_transform(Data[numerical_cols])
Data.head()

In [ ]:
# Task 6: Write your code here:

# Check target imbalance
target_dist = y.value_counts(normalize=True)
print(target_dist)

if target_dist.max() > 0.6:
    print("The target variable is imbalanced.")
else:
    print("The target variable is balanced.")

In [ ]:
# Task 1: Write your code here:

np.random.seed(42)

X_reg = pd.DataFrame({"Feature_1": np.random.randn(500)})
y_reg = 3 * X_reg["Feature_1"] + np.random.randn(500) * 0.5


X_clf = pd.DataFrame({"Feature_1": np.random.randn(500)})
y_clf = (X_clf["Feature_1"] > 0).astype(int)
X, y = X_clf.copy(), y_clf.copy()

# split ratio 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)
# Print shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Task 2
# K-Fold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

In [ ]:
#Task 3
# we need to train scaler
for tr_idx, va_idx in kf.split(X):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_va)

In [ ]:
mae_scores.append(mean_absolute_error(y_va, y_pred))

In [ ]:
#Task 5
print("Average MAE across folds:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:
# Feature importance

importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X_train.columns[indices], rotation=90)
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("Feature Importance from Random Forest Model")
plt.tight_layout()
plt.show()




In [ ]:
# Task 2: Write your code here:

# Predict delivery time
y_pred = model.predict(X_test)

plt.figure(figsize=(8, 5))
plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.title("Histogram of Predicted Delivery Time")
plt.show()

In [ ]:
# Task Bonus: Write your code here:
# need to install
!pip install catboost

X_np = X.values if hasattr(X, "values") else X
y_np = y.values if hasattr(y, "values") else y

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_maes = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np), start=1):
    X_train, X_val = X_np[train_idx], X_np[val_idx]
    y_train, y_val = y_np[train_idx], y_np[val_idx]

    # Model 1 Random Forest
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    # Model 2 CatBoost
    cb = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="MAE",
        random_seed=42,
        verbose=0
    )

    # Train  odels
    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    # Predict
    pred_rf = rf.predict(X_val)
    pred_cb = cb.predict(X_val)

    # Average predictions
    pred_avg = (pred_rf + pred_cb) / 2.0

    # MAE
    fold_mae = mean_absolute_error(y_val, pred_avg)
    fold_maes.append(fold_mae)

    print(f"Fold {fold} MAE (Ensemble Avg): {fold_mae:.4f}")

print("-" * 40)
print(f"Mean MAE (Ensemble Avg): {np.mean(fold_maes):.4f}")
print(f"Std  MAE (Ensemble Avg): {np.std(fold_maes):.4f}")
